# Modules import

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# %reload_ext autoreload
# %pip install holidays==0.36

In [3]:
import pandas as pd
from megatron.config import set_config

test = pd.read_csv(
    filepath_or_buffer="../data/raw/test.csv",
    parse_dates=["date"],
    index_col=["id"],
    engine="pyarrow",
)

max_date = test["date"].max()
min_date = max_date - pd.DateOffset(years=2)
fh_size = (max_date - test["date"].min()).days + 1

set_config(
    SEASONAL_PERIOD=7,
    MIN_LENGTH=30,
    COUNTRY="EC",
    MIN_DATE=min_date,
    MAX_DATE=max_date,
    FH_SIZE=fh_size,
    SEED=42,
)

import megatron.config as config
from megatron.transformers.additional import Mapper, InitialPreprocessing
from megatron.transformers.series import PlateauDetector
from megatron.visualization.series import seriesPlot

import plotly.express as px

px.defaults.width, px.defaults.height = config.FIG_WIDTH, config.FIG_HEIGHT  # type: ignore

# Exogenous features

## Oil

In [4]:
oil = (
    pd.read_csv(
        filepath_or_buffer="../data/raw/oil.csv", 
        index_col=["date"], 
        parse_dates=["date"]
    )
    .asfreq("D")
    .loc[min_date:]
    .interpolate(method="linear")
    .rename(columns={"dcoilwtico": "oil"})
)
oil.head()

,oil
date,
2015-08-31,49.20
2015-09-01,45.38
2015-09-02,46.30
2015-09-03,46.75
2015-09-04,46.02


In [5]:
seriesPlot(data=oil, title="Ecuador series")

## Stores

In [6]:
stores = (
    pd.read_csv("../data/raw/stores.csv", index_col=["store_nbr"])
    .drop(columns=["city", "state"])
    .astype({"cluster": float})
    .rename(columns={"cluster": "store_group"})
)
stores["type"] = stores["type"].factorize()[0].astype(float)
stores.head()

,type,store_group
store_nbr,,
1,0.0,13.0
2,0.0,13.0
3,0.0,8.0
4,0.0,9.0
5,0.0,4.0


## Transactions

In [7]:
transactions = (
    pd.read_csv(
        filepath_or_buffer="../data/raw/transactions.csv",
        index_col=["store_nbr", "date"],
        parse_dates=["date"],
        engine="pyarrow",
    )
    .groupby("store_nbr")
    .apply(lambda x: x.droplevel(["store_nbr"]).asfreq("D").loc[min_date:])
    .sort_index()
)
transactions.head()

transactions
store_nbr date                    
1         2015-08-31        1785.0
          2015-09-01        1769.0
          2015-09-02        1892.0
          2015-09-03        1744.0
          2015-09-04        1771.0

### Initial filters

In [8]:
processing = InitialPreprocessing(w=2 * config.SEASONAL_PERIOD)
transactions = processing.drop_zero_series(transactions)
transactions = processing.trim_leading_zeros(transactions)
transactions = processing.drop_trailing_zero_window_series(transactions)

### Plateau detection

In [9]:
seriesPlot(
    data=transactions,
    n_series=10,
    w=2 * config.SEASONAL_PERIOD,
    pld=True,
    seed=12,
)

In [10]:
transactions = (
    PlateauDetector(w=2 * config.SEASONAL_PERIOD, truncate=True)
    .fit_transform(transactions)
)

In [11]:
instance = 18
seriesPlot(
    data=transactions.loc[instance],
    w=2 * config.SEASONAL_PERIOD,
    title=f"Transactions in Equador for {instance}",
)

# Sales

In [12]:
sales = (
    pd.read_csv(
        filepath_or_buffer="../data/raw/train.csv",
        index_col=["store_nbr", "family", "date"],
        parse_dates=["date"],
        engine="pyarrow",
    )
    .groupby(["store_nbr", "family"])
    .apply(lambda x: x.droplevel(["store_nbr", "family"]).asfreq("D").loc[min_date:])
    .sort_index()
)

X_exog_train = (
    sales[["onpromotion"]]
    .fillna(0)
    .astype(float)
    .rename(columns={"onpromotion": "on_promotion"})
)

X_exog_test = (
    test.set_index(["store_nbr", "family", "date"])
    .astype(float)
    .rename(columns={"onpromotion": "on_promotion"})
    .sort_index()
)

sales = sales[["sales"]]
sales.head()

sales
store_nbr family     date             
1         AUTOMOTIVE 2015-08-31    6.0
                     2015-09-01    1.0
                     2015-09-02    4.0
                     2015-09-03    2.0
                     2015-09-04    0.0

In [15]:
mapper = Mapper()
sales = mapper.fit_transform(sales)
sales.head()

sales
index date             
0     2015-08-31    6.0
      2015-09-01    1.0
      2015-09-02    4.0
      2015-09-03    2.0
      2015-09-04    0.0

## Initial filters

In [16]:
processing = InitialPreprocessing(w=2 * config.SEASONAL_PERIOD)
sales = processing.drop_zero_series(sales)
sales = processing.trim_leading_zeros(sales)
sales = processing.drop_trailing_zero_window_series(sales)

In [17]:
sales = mapper.inverse_transform(sales)

## Plateau detection

In [18]:
seriesPlot(
    data=sales.loc[[18, 25]].sort_index(),
    n_series=10,
    w=2 * config.SEASONAL_PERIOD,
    pld=True,
    pd_value=0,
    seed=22,
)

In [19]:
sales = sales.join(transactions[[]], how="inner")

In [20]:
seriesPlot(
    data=sales.loc[[18, 25]].sort_index(),
    n_series=10,
    w=2 * config.SEASONAL_PERIOD,
    pld=True,
    pd_value=0,
    seed=22,
)

# Saving

In [21]:
X_exog_train = X_exog_train.join(sales[[]], how="inner")
X_exog_test = X_exog_test.join(
    sales[~sales.droplevel(-1).index.duplicated(keep="first")][[]].droplevel(-1),
    how="inner",
)

In [22]:
oil.to_parquet("../data/processed/oil.pq")
stores.to_parquet("../data/processed/stores.pq")
transactions.to_parquet("../data/processed/transactions.pq")
sales.to_parquet("../data/processed/sales.pq")
X_exog_train.to_parquet("../data/processed/sales_exog_train.pq")
X_exog_test.to_parquet("../data/processed/sales_exog_test.pq")